# Практика · GPT-родина

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

> ⏱ Зошит **навчає одну авторегресійну модель** і показує на ній пʼять способів
> породження тексту. Заміряно: **близько півтори хвилини процесорного часу**
> (90 с у трьох прогонах поспіль) на чотирьох ядрах без відеокарти, **в один
> потік**. Від першої клітинки до останньої минає близько двох хвилин, а на
> завантаженій машині більше. Зошит друкує і навантаження, і власний
> процесорний час останньою клітинкою, тож ці числа можна звірити.

## Задача зошита

У нас є справжній український текст без жодних міток — переклади інтерфейсів,
які лежать у системі. Ми **навчимо на ньому маленький GPT** і подивимось, що з
такою моделлю можна робити:

1. напишемо **причинну маску руками** й звіримо свою увагу з бібліотечною;
2. навчимо модель і перевіримо, що вона **пройшла уніграмний рубіж** — тобто
   справді бачить контекст, а не вгадує частотне слово;
3. **дістанемо з неї текст** пʼятьма способами: жадібно, з температурою,
   через `top-k`, через `top-p` і променем;
4. спробуємо **промпт замість донавчання** й чесно подивимось, чи він працює
   на такому масштабі.

⚠️ **Готових ваг GPT у нас немає** — мережі в зошитах курсу заборонено, а кеш
порожній. Тому модель ми навчаємо самі. Це навчальний макет на два з гаком
мільйони ваг; від нього не варто чекати осмисленого тексту, і в кінці зошита
сказано, чого саме чекати не варто.

## 0 · Середовище й чому ми фіксуємо потоки

Зошит міряє час, а час на спільній машині бреше двома способами.

**Стінний годинник** показує, скільки минуло реального часу. Якщо поруч
рахується щось іще, він покаже більше — і число нічого не варте.

**Процесорний годинник** (`time.process_time()`) рахує лише той час, коли
процесор працював над нашою програмою. Але й він має пастку: якщо бібліотека
лінійної алгебри розкладає роботу на кілька потоків, то потоки, які **чекають**,
теж рахуються як робота. Тому спершу фіксуємо один потік — і робимо це **до**
імпорту `numpy`, бо змінні середовища читаються при завантаженні бібліотеки.

In [ ]:
import os
# ⚠️ ці три рядки мусять стояти ДО імпорту numpy і torch — інакше бібліотеки
# вже запустять свої потоки, і процесорний час стане брехливим
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import glob, gettext, re, math, gc, time, sys
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)
START = time.process_time()          # звідси рахуємо власний час зошита

print('python              ', sys.version.split()[0])
print('torch               ', torch.__version__)
print('ядер у машині       ', os.cpu_count())
print('потоків у torch     ', torch.get_num_threads())
print('відеокарта          ', 'є' if torch.cuda.is_available() else 'немає')
print('навантаження машини ', round(os.getloadavg()[0], 2))

## 1 · Корпус

Беремо українські переклади інтерфейсів із `/usr/share/locale/uk/LC_MESSAGES/`.
Це справжня українська мова, вона вже лежить на машині, і вона однакова від
запуску до запуску.

Кожен запис каталогу — пара «англійський оригінал → український переклад».
Модель учиться на **українському** боці; англійський знадобиться в самому кінці,
коли ми даватимемо моделі задачу класифікації.

Ділити текст на слова будемо канонічним для курсу регулярним виразом, у якому
апостроф — звʼязка всередині слова, а не роздільник.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
ERROR_WORDS = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)

def load_corpus():
    '''Повертає трійки (програма, англійський оригінал, слова перекладу).'''
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                    # зламаний каталог просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                out.append((program, source, re.findall(TOKEN_PATTERN, target.lower())))
    return out

corpus = load_corpus()
if len(corpus) < 1000:
    raise RuntimeError('українських каталогів у системі майже немає — '
                       'зошитові немає на чому працювати')

# надто короткі й надто довгі рядки прибираємо: у рядку з одного слова нічого
# передбачати, а хвіст на півтисячі слів роздув би пачки самим заповнювачем
rows = [r for r in corpus if 2 <= len(r[2]) <= 30]
lengths = np.array([len(r[2]) for r in rows])

print('записів у каталогах   ', len(corpus))
print('лишилось після відсіву', len(rows))
print('слововживань          ', int(lengths.sum()))
print('різних словоформ      ', len({w for r in rows for w in r[2]}))
print('медіана довжини       ', int(np.median(lengths)), 'слів')
print('програм               ', len({r[0] for r in rows}))

**Запамʼятай медіану.** Половина рядків коротша за неї — а це означає, що наша
модель бачитиме дуже короткі послідовності. Далі це пояснить, чому породжений
текст такий короткий.

## 2 · Три частини: навчальна, відкладена, перевірна

Ділимо на **три** частини, а не на дві:

- **навчальна** — на ній модель учиться;
- **відкладена** — на ній добирають гіперпараметри. Швидкість навчання ми
  дібрали окремим прогоном саме на ній (сітка з пʼяти значень, мінімум усередині
  діапазону, а не скраю), тож тут вона потрібна лише для того, щоб поділ був
  такий самий, як у того добору;
- **перевірна** — на ній оголошується результат. Модель не бачить її ніколи.

⚠️ Рядки лежать **у порядку програм**: спершу всі рядки однієї програми, потім
іншої. Якби ми взяли «перші десять відсотків», це була б не менша вибірка, а
**вужчий домен**. Тому перемішуємо з фіксованим зерном.

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(rows))
n_test = len(rows) // 10
test_positions = set(order[:n_test].tolist())

train_all = [rows[i] for i in range(len(rows)) if i not in test_positions]
test_rows = [rows[i] for i in range(len(rows)) if i in test_positions]

# відкладена — 5 % навчальної частини, окреме зерно
rng_dev = np.random.default_rng(1)
order_dev = rng_dev.permutation(len(train_all))
n_dev = int(round(len(train_all) * 0.05))
dev_positions = set(order_dev[:n_dev].tolist())

dev_rows = [train_all[i] for i in range(len(train_all)) if i in dev_positions]
train_rows = [train_all[i] for i in range(len(train_all)) if i not in dev_positions]

print('навчальних ', len(train_rows))
print('відкладених', len(dev_rows))
print('перевірних ', len(test_rows))
print('слововживань у навчальній частині', sum(len(r[2]) for r in train_rows))

## 3 · Словник і спецтокени

Модель працює з номерами, а не зі словами. Беремо всі слова, що трапились у
навчальній частині щонайменше пʼять разів, і додаємо чотири службові позиції:

- `[PAD]` — заповнювач, щоб рядки різної довжини склались у прямокутну пачку;
- `[BOS]` — «початок рядка». Без нього перше справжнє слово нічим передбачати:
  ліворуч від нього порожньо;
- `[EOS]` — «кінець рядка». Модель мусить уміти сказати, що речення скінчилось,
  інакше породження ніколи не зупиниться;
- `[UNK]` — усе, чого немає в словнику.

In [ ]:
PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIAL = ['[PAD]', '[BOS]', '[EOS]', '[UNK]']
MAXLEN = 32                                # вікно моделі: стільки позицій вона бачить

counts = Counter(w for r in train_rows for w in r[2])
itos = SPECIAL + [w for w, c in counts.most_common() if c >= 5]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)

def encode(rows_list):
    '''Кожен рядок -> [BOS] слова [EOS], доповнений PAD до MAXLEN.'''
    out = np.zeros((len(rows_list), MAXLEN), dtype=np.int64)
    for i, r in enumerate(rows_list):
        ids = [BOS] + [stoi.get(w, UNK) for w in r[2][:MAXLEN - 2]] + [EOS]
        out[i, :len(ids)] = ids
    return torch.from_numpy(out)

X_train, X_dev, X_test = encode(train_rows), encode(dev_rows), encode(test_rows)

covered = sum(counts[w] for w in itos[len(SPECIAL):])
print('слів у словнику    ', VOCAB)
print('покрито слововживань', round(covered / sum(counts.values()), 4))
print('форма навчальної пачки', tuple(X_train.shape))
print('перший рядок як номери:', X_train[0][:10].tolist())
print('він же словами        :', ' '.join(itos[i] for i in X_train[0][:10].tolist()))

## 4 · Причинна маска руками

Перш ніж будувати модель, напишемо серце авторегресії — **самоувагу з причинною
маскою** — власними руками й звіримо з бібліотечною. Це найкорисніша перевірка в
зошиті: усередині `nn.MultiheadAttention` немає магії, там пʼять рядків.

Що ми робимо. З кожного вектора позиції дістають три інші: **запит** `Q`, **ключ**
`K` і **значення** `V`. Оцінка того, наскільки позиція `i` цікавиться позицією
`j`, — це скалярний добуток `Q[i]·K[j]`, поділений на корінь із розміру. Далі
додаємо маску: нуль там, де дивитись можна, і мінус нескінченність там, де не
можна. Після `softmax` заборонені місця стають рівно нулем.

In [ ]:
torch.manual_seed(0)
d_model, n_pos = 16, 6
x = torch.randn(1, n_pos, d_model)

# причинна маска: True там, де дивитись ЗАБОРОНЕНО (тобто вище діагоналі)
forbidden = torch.triu(torch.ones(n_pos, n_pos, dtype=torch.bool), diagonal=1)
print('маска (True = заборонено):')
print(forbidden.int().numpy())

attention = nn.MultiheadAttention(d_model, num_heads=1, batch_first=True, bias=False)
library_out, _ = attention(x, x, x, attn_mask=forbidden, need_weights=False)
print('\nвихід бібліотеки, перша позиція:', library_out[0, 0, :4].detach().numpy().round(5))

Тепер те саме своїми руками. Ваги беремо з тієї самої бібліотечної голови, щоб
порівнювати обрахунок, а не випадкову ініціалізацію.

In [ ]:
# in_proj_weight складено вертикально: спершу ваги Q, потім K, потім V
Wq, Wk, Wv = attention.in_proj_weight.chunk(3, dim=0)
Wo = attention.out_proj.weight

Q = x @ Wq.T                       # (1, позиції, d)
K = x @ Wk.T
V = x @ Wv.T

scores = Q @ K.transpose(1, 2) / math.sqrt(d_model)
scores = scores.masked_fill(forbidden, float('-inf'))   # ось і вся причинність
weights = torch.softmax(scores, dim=-1)
our_out = (weights @ V) @ Wo.T

print('ваги уваги для першої позиції:', weights[0, 0].detach().numpy().round(4))
print('ваги уваги для останньої      :', weights[0, -1].detach().numpy().round(4))
print()
print('наш вихід, перша позиція:', our_out[0, 0, :4].detach().numpy().round(5))

difference = float((our_out - library_out).abs().max())
print('\nмакс. розбіжність із бібліотекою', '%.3e' % difference)
print('машинний епсилон float32        ', '%.3e' % float(np.finfo(np.float32).eps))
assert difference < float(np.finfo(np.float32).eps), 'розрахунок розійшовся!'
print('✅ збігається з точністю до одиниці останнього розряду')

Зверни увагу на перший рядок ваг уваги: там **одиниця й нулі**. Перша позиція
має право дивитись лише на себе, тож `softmax` від одного дозволеного числа
завжди дає одиницю. В останньому рядку — шість ненульових ваг: він бачить усе.

## 5 · Модель

Це звичайний стос енкодерів трансформера, який стає **декодером** рівно тому, що
ми передаємо йому причинну маску. Архітектурно нічого нового.

In [ ]:
class TinyGPT(nn.Module):
    '''Авторегресійна мовна модель: ембединги, стос блоків, голова на словник.'''

    def __init__(self, vocab, d=128, layers=2, heads=2, ff=256, dropout=0.1):
        super().__init__()
        self.token = nn.Embedding(vocab, d, padding_idx=PAD)
        self.position = nn.Embedding(MAXLEN, d)   # навчувані позиції
        self.drop = nn.Dropout(dropout)
        block = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=ff,
                                           dropout=dropout, batch_first=True,
                                           norm_first=True, activation='gelu')
        self.blocks = nn.TransformerEncoder(block, num_layers=layers)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)

    def body(self, ids):
        n = ids.size(1)
        h = self.token(ids) + self.position(torch.arange(n))
        h = self.drop(h)
        # ось те, заради чого вся тема: позиція t не бачить нічого правіше себе
        causal = torch.triu(torch.full((n, n), float('-inf')), diagonal=1)
        h = self.blocks(h, mask=causal, src_key_padding_mask=(ids == PAD))
        return self.norm(h)

    def forward(self, ids):
        return self.head(self.body(ids))

model = TinyGPT(VOCAB)
print('ваг у моделі        ', sum(p.numel() for p in model.parameters()))
print('з них у ембедингах  ', model.token.weight.numel())
print('з них у голові      ', model.head.weight.numel() + model.head.bias.numel())

Більшість ваг сидить у двох таблицях на словник — вхідній і вихідній. Саме тому
розмір словника впливає на вагу моделі сильніше, ніж кількість шарів.

## 6 · Навчання

Кожна позиція вгадує наступну. Втрата — перехресна ентропія, заповнювач із неї
викинуто. Голову на словник рахуємо **лише на позиціях із ціллю**: це те саме
число, але вчетверо дешевше, бо решта позицій — заповнювач.

In [ ]:
# ⚠️ Швидкість 0.004 дібрано окремим прогоном на ВІДКЛАДЕНІЙ вибірці (сітка з
# пʼяти значень, мінімум усередині) — але на довшому бюджеті, 600 кроків. Тут ми
# вкладаємось у час і беремо 320. Коротший бюджет зазвичай любить трохи більший
# крок, тож наше значення радше обережне, ніж оптимальне.
STEPS, BATCH, LR = 320, 64, 0.004

def batch_loss(net, batch):
    '''Втрата на одній пачці: кожна позиція вгадує наступну.'''
    # обрізаємо стовпці-заповнювачі: медіана рядка шість слів, а вікно 32
    width = int((batch != PAD).sum(0).nonzero().max()) + 1
    batch = batch[:, :width]
    hidden = net.body(batch[:, :-1])
    target = batch[:, 1:]
    live = target != PAD
    return F.cross_entropy(net.head(hidden[live]), target[live])

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR,
                                               total_steps=STEPS, pct_start=0.1)
generator = torch.Generator().manual_seed(0)
torch.manual_seed(0)

t0 = time.process_time()
model.train()
history = []
for step in range(1, STEPS + 1):
    idx = torch.randint(0, X_train.size(0), (BATCH,), generator=generator)
    loss = batch_loss(model, X_train[idx])
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    schedule.step()
    if step % 40 == 0:
        history.append((step, round(float(loss), 4)))

print('кроків       ', STEPS)
print('процесорних с', round(time.process_time() - t0, 1))
print('втрата по кроках (на навчальній пачці):')
for step, value in history:
    print('   крок %4d   %.4f' % (step, value))

## 7 · Чи модель узагалі щось вивчила

Перед будь-яким висновком треба перевірити, що модель **пройшла уніграмний
рубіж** — рівень моделі, яка не дивиться на сусідів зовсім, а щоразу називає
найчастіші слова корпусу. Поки цей рубіж не пройдено, модель контексту не бачить,
і все, що ми з неї дістанемо, буде шумом.

In [ ]:
def unigram_loss(train_source, eval_source):
    '''Втрата моделі, яка не дивиться на сусідів (нати на слово).'''
    counter, total = Counter(), 0
    for r in train_source:
        ids = [stoi.get(w, UNK) for w in r[2][:MAXLEN - 2]] + [EOS]
        counter.update(ids)
        total += len(ids)
    probs = np.full(VOCAB, 1.0)               # добавка 1 на невидані токени
    for key, value in counter.items():
        probs[key] += value
    probs /= probs.sum()
    loss_sum, count = 0.0, 0
    for r in eval_source:
        ids = [stoi.get(w, UNK) for w in r[2][:MAXLEN - 2]] + [EOS]
        loss_sum += -np.log(probs[ids]).sum()
        count += len(ids)
    return loss_sum / count

@torch.no_grad()
def model_loss(net, data, chunk=128):
    '''Середня втрата моделі на передбаченні наступного слова.'''
    net.eval()
    total, count = 0.0, 0
    for i in range(0, data.size(0), chunk):
        batch = data[i:i + chunk]
        width = int((batch != PAD).sum(0).nonzero().max()) + 1
        batch = batch[:, :width]
        hidden = net.body(batch[:, :-1])
        target = batch[:, 1:]
        live = target != PAD
        total += float(F.cross_entropy(net.head(hidden[live]), target[live],
                                       reduction='sum'))
        count += int(live.sum())
    return total / count

bound = unigram_loss(train_rows, test_rows)
ours = model_loss(model, X_test)
print('уніграмний рубіж, нати   %.4f  (перплексія %8.2f)' % (bound, math.exp(bound)))
print('наша модель,      нати   %.4f  (перплексія %8.2f)' % (ours, math.exp(ours)))
print()
if ours < bound:
    print('✅ рубіж пройдено: модель бачить контекст, а не лише частоти')
else:
    print('❌ рубіж НЕ пройдено: усе, що нижче, вимірювало б шум')

## 8 · Породження: жадібне

Тепер найцікавіше. Модель віддає не слово, а **розподіл** — число для кожного
слова словника. Найпростіше правило: узяти найімовірніше. Дописуємо слово до
входу й питаємо знову, доки не випаде `[EOS]` або не скінчиться вікно.

In [ ]:
@torch.no_grad()
def next_distribution(net, ids):
    '''Розподіл наступного слова для однієї послідовності номерів.'''
    net.eval()
    logits = net(torch.tensor([ids]))[0, -1]
    logits[PAD] = -1e9
    logits[BOS] = -1e9              # ці два токени породжувати безглуздо
    return logits

@torch.no_grad()
def generate(net, prefix_words, rule='greedy', temperature=1.0, top_k=0, top_p=0.0,
             seed=0, max_new=14):
    '''Дописує продовження до префікса за одним із правил відбору.'''
    gen = torch.Generator().manual_seed(seed)
    ids = [BOS] + [stoi.get(w, UNK) for w in prefix_words]
    produced = []
    for _ in range(max_new):
        logits = next_distribution(net, ids[-MAXLEN:])
        if rule == 'greedy':
            choice = int(logits.argmax())
        else:
            scaled = logits / temperature
            if top_k:
                threshold = scaled.topk(top_k).values[-1]
                scaled = scaled.masked_fill(scaled < threshold, -1e9)
            if top_p:
                order = scaled.argsort(descending=True)
                probs = torch.softmax(scaled[order], -1)
                keep = (probs.cumsum(0) - probs) < top_p
                blocked = order[~keep]
                scaled = scaled.clone()
                scaled[blocked] = -1e9
            choice = int(torch.multinomial(torch.softmax(scaled, -1), 1, generator=gen))
        if choice == EOS:
            break
        produced.append(choice)
        ids.append(choice)
    return produced

PREFIXES = [['не', 'вдалося'], ['помилка', 'під'], ['файл'], ['неможливо']]
for prefix in PREFIXES:
    tail = generate(model, prefix, 'greedy')
    print('%-18s -> %s' % (' '.join(prefix), ' '.join(itos[i] for i in tail)))

Текст виглядає як уривок повідомлення інтерфейсу — бо саме на них модель і
вчилась. Але жадібне правило детерміноване: той самий промпт завжди дає ту саму
відповідь, і відповідь ця часто йде по колу.

## 9 · Температура

Ділимо оцінки моделі на число `T` перед `softmax`. Менше за одиницю — розподіл
стає гострішим; більше — пласкішим. Порахуємо не тільки текст, а й **скільки
різних слів** модель випустила на тридцяти породженнях: це проста міра
різноманіття.

In [ ]:
def diversity(net, temperature, top_k=0, top_p=0.0, runs=30):
    '''Частка різних слів серед породжених — груба міра різноманіття.'''
    produced = []
    for seed in range(runs):
        produced += generate(net, ['не', 'вдалося'], 'sample', temperature,
                             top_k, top_p, seed=seed)
    return len(set(produced)) / max(len(produced), 1), len(produced) / runs

print('%-16s %10s %10s' % ('температура', 'різних', 'слів'))
for temperature in [0.5, 0.8, 1.0, 1.5]:
    share, length = diversity(model, temperature)
    print('%-16.1f %10.4f %10.2f' % (temperature, share, length))

print()
print('приклади при різних температурах (зерно 3):')
for temperature in [0.5, 1.0, 1.5]:
    tail = generate(model, ['не', 'вдалося'], 'sample', temperature, seed=3)
    print('  T = %.1f  не вдалося %s' % (temperature, ' '.join(itos[i] for i in tail)))

## 10 · `top-k` і `top-p`

Температура тягне **весь** розподіл одразу: піднявши її заради різноманіття, ти
піднімаєш і ймовірність тисяч безглуздих слів у хвості. Два інші правила чіпають
не висоти, а **склад** кандидатів.

`top-k` лишає рівно `k` найкращих. `top-p` лишає стільки найкращих, скільки
треба, щоб їхня сума вперше перевищила `p`. Різниця найкраще видно на двох
різних ситуаціях: коли модель упевнена й коли вагається.

In [ ]:
@torch.no_grad()
def candidates_for(prefix_words, p=0.9, k=40):
    '''Скільки кандидатів лишає кожне правило на конкретному префіксі.'''
    ids = [BOS] + [stoi.get(w, UNK) for w in prefix_words]
    probs = torch.softmax(next_distribution(model, ids), -1)
    ordered = probs.sort(descending=True).values
    entropy = float(-(probs * probs.clamp(min=1e-12).log()).sum())
    n_for_p = int((ordered.cumsum(0) < p).sum()) + 1
    return dict(перплексія=round(math.exp(entropy), 2),
                найкращий=round(float(ordered[0]), 4),
                top_k=k, top_p_дає=n_for_p,
                маса_top_k=round(float(ordered[:k].sum()), 4))

for prefix in [['не', 'вдалося'], ['у'], ['помилка']]:
    print('%-16s %s' % (' '.join(prefix), candidates_for(prefix)))

Дивись на колонку `top_p_дає`: одне й те саме правило набирає різну кількість
кандидатів залежно від того, наскільки модель упевнена. `top-k` завжди бере
сорок — і там, де їх забагато, і там, де замало. У цьому вся різниця.

In [ ]:
print('%-24s %10s %10s' % ('правило', 'різних', 'слів'))
share, length = diversity(model, 1.0)
print('%-24s %10.4f %10.2f' % ('температура 1.0', share, length))
share, length = diversity(model, 1.0, top_k=5)
print('%-24s %10.4f %10.2f' % ('top-k = 5', share, length))
share, length = diversity(model, 1.0, top_k=40)
print('%-24s %10.4f %10.2f' % ('top-k = 40', share, length))
share, length = diversity(model, 1.0, top_p=0.9)
print('%-24s %10.4f %10.2f' % ('top-p = 0.9', share, length))

## 11 · Промінь і його пастка

Промінь тримає одночасно `k` найкращих незавершених версій. Версії порівнюють за
сумою логарифмів імовірностей. Логарифм числа, меншого за одиницю, відʼємний —
тож сума **завжди спадає** з кожним новим словом, і з двох версій промінь майже
завжди обирає коротшу.

Перевіримо це числом на однаковому для всіх ширин зразку перевірних рядків.

In [ ]:
@torch.no_grad()
def beam(net, prefix_ids, width, alpha=0.0, max_new=12):
    '''Промінь ширини width. alpha — степінь, на який ділять оцінку.'''
    def score(item):
        length = max(len(item[2]), 1)
        return item[0] / (length ** alpha) if alpha else item[0]

    beams = [(0.0, list(prefix_ids), [], False)]
    for _ in range(max_new):
        alive = [b for b in beams if not b[3]]
        if not alive:
            break
        candidates = [b for b in beams if b[3]]
        for item in alive:
            logprob = torch.log_softmax(next_distribution(net, item[1][-MAXLEN:]), -1)
            best = logprob.topk(width)
            for j in range(width):
                token = int(best.indices[j])
                candidates.append((item[0] + float(best.values[j]),
                                   item[1] + [token], item[2] + [token], token == EOS))
        candidates.sort(key=score, reverse=True)
        beams = candidates[:width]
    return max(beams, key=score)[2]

sample = [r for r in test_rows if len(r[2]) >= 5][:60]
prefixes = [[BOS] + [stoi.get(w, UNK) for w in r[2][:3]] for r in sample]
reference = float(np.mean([len(r[2]) - 3 for r in sample]))

t0 = time.process_time()
print('%-24s %12s' % ('спосіб', 'слів у виході'))
for width in [1, 2, 4, 8]:
    lengths_out = [len([t for t in beam(model, p, width) if t != EOS]) for p in prefixes]
    print('%-24s %12.2f' % ('промінь %d' % width, float(np.mean(lengths_out))))
for width in [4, 8]:
    lengths_out = [len([t for t in beam(model, p, width, alpha=0.7) if t != EOS])
                   for p in prefixes]
    print('%-24s %12.2f' % ('промінь %d, alpha 0.7' % width, float(np.mean(lengths_out))))
print('%-24s %12.2f' % ('справжнє продовження', reference))
print('\nпроцесорних с на промінь', round(time.process_time() - t0, 1))

Довжина падає з шириною — і піднімається назад, щойно ми ділимо оцінку на
довжину. Це і є та пастка: ширший пошук сам собою робить **гірше**, а не краще.

## 12 · Промпт замість донавчання

Остання спроба. Покладемо задачу «чи повідомляє цей рядок про помилку» просто у
**вхід** моделі: кілька показових пар «текст → слово-мітка», а далі запит. Ваги
не чіпаємо взагалі. Читаємо, яке зі слів-міток модель уважає ймовірнішим.

Мітку беремо з **англійського** боку каталогу, якого модель не бачила ніколи, —
тож вона не кругова.

⚠️ І одразу назвемо межу, яку тут не обійти: вікно нашої моделі — **32 позиції**.
Чотири показові приклади разом із запитом у нього ледве влазять, а більше вже ні.
У великих моделях вікно в десятки разів довше, і саме тому туди вміщається сотня
прикладів. Це перша з двох причин, чому промпт на нашому масштабі не працює;
друга — сам корпус, у якому візерунка «список пар» просто немає.

In [ ]:
YES, NO = 'помилка', 'гаразд'
print('«%s» у словнику: %s · «%s» у словнику: %s'
      % (YES, YES in stoi, NO, NO in stoi))

labels_train = np.array([1 if ERROR_WORDS.search(r[1]) else 0 for r in train_rows])
queries = test_rows[:400]
labels_query = np.array([1 if ERROR_WORDS.search(r[1]) else 0 for r in queries])

positive = [i for i in range(len(train_rows)) if labels_train[i] == 1]
negative = [i for i in range(len(train_rows)) if labels_train[i] == 0]

@torch.no_grad()
def prompt_accuracy(n_demo, seed):
    '''Частка правильних відповідей, коли задача описана самим входом.'''
    draw = np.random.default_rng(seed)
    right = 0
    for position, row in enumerate(queries):
        ids = [BOS]
        picks = ([(positive[j], 1) for j in draw.integers(0, len(positive), n_demo // 2)]
                 + [(negative[j], 0) for j in draw.integers(0, len(negative),
                                                            n_demo - n_demo // 2)])
        for index, label in picks:
            ids += [stoi.get(w, UNK) for w in train_rows[index][2][:6]]
            ids += [stoi[YES] if label else stoi[NO]]
        ids += [stoi.get(w, UNK) for w in row[2][:8]]
        # ⚠️ вікно моделі — MAXLEN позицій. Довший промпт у нього просто не влазить,
        # і це не наша недбалість, а межа макета: у великих моделей вікно в десятки
        # разів довше, і саме тому туди вміщається сотня показових прикладів
        logits = next_distribution(model, ids[-MAXLEN:])
        guess = 1 if float(logits[stoi[YES]]) > float(logits[stoi[NO]]) else 0
        right += int(guess == labels_query[position])
    return right / len(queries)

print()
print('постійна відповідь «не помилка» %.4f' % float((labels_query == 0).mean()))
for n_demo in [0, 2, 4]:
    values = [prompt_accuracy(n_demo, s) for s in [0, 1, 2]]
    print('показових прикладів %d      %.4f  (три набори: %s)'
          % (n_demo, float(np.median(values)), [round(v, 4) for v in values]))

## 13 · А що вміє те саме подання, якщо його просто взяти

Промпт не спрацював — але це не означає, що модель нічого не вивчила.
Заморозимо її тіло, візьмемо середній вектор по позиціях і навчимо поверх
звичайну логістичну регресію. Ваги моделі при цьому **не міняються** — ми лише
читаємо те, що вона вже вміє.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

@torch.no_grad()
def features(net, data, chunk=128):
    '''Заморожене подання рядка: середнє по справжніх позиціях.'''
    net.eval()
    out = []
    for i in range(0, data.size(0), chunk):
        batch = data[i:i + chunk]
        hidden = net.body(batch)
        mask = (batch != PAD).float().unsqueeze(-1)
        out.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).numpy())
    return np.concatenate(out)

N_PROBE = 12000
F_train = features(model, X_train[:N_PROBE])
F_test = features(model, X_test[:len(queries)])

probe = LogisticRegression(max_iter=2000).fit(F_train, labels_train[:N_PROBE])
predicted = probe.predict(F_test)
print('заморожений зонд: точність %.4f · F1 класу «помилка» %.4f'
      % (accuracy_score(labels_query, predicted), f1_score(labels_query, predicted)))
print()
print('Ті самі ваги, та сама модель. Різниця лише в тому, ЯК ми до неї звертаємось:')
print('через текст на вході — не працює; через прочитане подання — працює.')

## 14 · Скільки це коштувало й чого не варто чекати

Останнє — час. Друкуємо процесорний, бо стінний на спільній машині бреше.

In [ ]:
del probe, F_train, F_test
gc.collect()

print('процесорний час зошита, с', round(time.process_time() - START, 1))
print('навантаження машини зараз', round(os.getloadavg()[0], 2))
print()
print('Чого від цього макета не варто чекати:')
print(' · осмисленого тексту — словник', VOCAB, 'слів, корпус із рядків інтерфейсів;')
print(' · навчання з промпту — заміряно вище, він не додає нічого;')
print(' · стабільних абсолютних чисел — корпус береться з системи читача,')
print('   а склад установлених програм у кожного свій.')
print()
print('Чого варто: усі механізми тут справжні. Причинна маска, температура,')
print('top-k, top-p і промінь працюють так само, як у великих моделях.')

## Завдання

### 🟢 Рівень 1
Додай до правил відбору **комбінацію** `top-k` разом із температурою (наприклад,
`k = 10`, `T = 0.8`) і зміряй її різноманіття тією самою функцією `diversity`.
**Зроблено, якщо** в таблиці зʼявився рядок із числом, і ти можеш сказати
словами, між якими двома чистими правилами він опинився.

### 🟡 Рівень 2
Промінь у розділі 11 міряв лише **довжину**. Додай другу метрику — **частку
вгаданих слів**: порівняй породжене з тим, що в перевірному рядку справді
стояло після третього слова. **Зроблено, якщо** ти можеш сказати, чи повторився
результат теми: ширший промінь без поділу на довжину робить гірше.

### 🔴 Рівень 3
Навчи **другу** модель із синусоїдальними позиціями замість навчуваних
(таблицю синусів побудуй сам) і прибери причинну маску в обох. **Зроблено,
якщо** ти дістав чотири числа перплексії й показав, що обвал без маски
залежить від того, чим задано позиції.